# Program 05: experience replay

**What:** keep a small memory buffer of old images (2,000) and mix them into every training batch.

**How:** fill the buffer with reservoir sampling so every image seen has an equal chance of being kept. Each step, compute the loss on the new batch plus a buffer batch.

**Why:** replay is the strongest simple approach. The buffer is also central to our problem: UnCLe found that old buffer data is one cause of relapse.

**Understanding note:** replay works because the model keeps seeing old data. That means any data still in the buffer can *teach the model again*, including data you were asked to delete.

**Done when:** average accuracy across all tasks clearly beats programs 03 and 04.

## 1. Setup
Same settings as programs 03 and 04, plus the buffer. 2,000 images is 4% of CIFAR-100's 50,000 training images.

In [1]:
import sys, time
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "common").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))  # lets us import common/

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

# Reload the shared code, so edits to common/ are picked up even without restarting the kernel.
import importlib
import common.utils, common.data, common.models, common.train
for module in (common.utils, common.data, common.models, common.train):
    importlib.reload(module)

from common.data import N_TASKS, SplitCIFAR100, task_classes
from common.models import resnet18_cifar
from common.train import average_accuracy, evaluate_tasks, forgetting, new_acc_matrix
from common.utils import RESULTS_DIR, get_device, log_run, set_seed

SETTINGS = {
    "epochs": 5, "lr": 0.03, "batch_size": 32, "seed": 0, "workers": 2,
    "buffer_size": 2000,     # how many old images we may keep
    "replay_batch": 32,      # buffer images mixed into every step
}
set_seed(SETTINGS["seed"])
device = get_device()
print("device:", device)

device: cuda


## 2. Data and model
Same as before: one ResNet-18, one 100-way output, class-incremental.

In [2]:
data = SplitCIFAR100(seed=SETTINGS["seed"], batch_size=SETTINGS["batch_size"], num_workers=SETTINGS["workers"])
model = resnet18_cifar(num_classes=100).to(device)
scaler = torch.amp.GradScaler(enabled=device.type == "cuda")

## 3. The buffer, and reservoir sampling

**What the buffer stores.** For each kept image: its **index** in the CIFAR-100 training set, and its label. Keeping the index is the same as keeping a copy of the raw image, because we only ever load images whose index is in the buffer. It has two advantages:
- When we replay an image, we load it fresh with a **new random crop and flip**, so 2,000 stored images look like many more.
- Later, for unlearning, we can see exactly **which images of which class** are still in the buffer (section 8, and program 17).

**Reservoir sampling.** The buffer holds 2,000 images, but over 10 tasks the model sees 50,000. Which ones should we keep? Reservoir sampling keeps a **uniform random sample of everything seen so far**, without knowing in advance how many images will come:

- While the buffer isn't full: keep every image.
- After that, for the n-th image seen: pick a random number j from 0 to n−1. If j < 2,000, the new image replaces slot j; otherwise drop it.

The new image is kept with probability 2,000/n, and each image already in the buffer survives with the same overall chance. So after task t, each of the t+1 tasks has about 2,000/(t+1) images in the buffer: 1,000 each after task 1, 200 each after task 9. You can check this in section 5's printout.

No task boundaries are needed, and older tasks are never favoured or pushed out unfairly.

In [3]:
class ReservoirBuffer:
    def __init__(self, size):
        self.size = size
        self.indices = np.zeros(size, dtype=np.int64)   # dataset index of each kept image
        self.labels = np.zeros(size, dtype=np.int64)
        self.n_kept = 0     # how many slots are filled
        self.n_seen = 0     # how many images have been offered so far

    def __len__(self):
        return self.n_kept

    def add(self, indices, labels):
        for idx, label in zip(indices.tolist(), labels.tolist()):
            if self.n_kept < self.size:                 # not full yet: keep it
                slot = self.n_kept
                self.n_kept += 1
            else:                                       # full: keep with chance size / (n_seen + 1)
                slot = np.random.randint(0, self.n_seen + 1)
                if slot >= self.size:
                    slot = None
            if slot is not None:
                self.indices[slot], self.labels[slot] = idx, label
            self.n_seen += 1

    def sample(self, n):
        """n random buffer images, freshly augmented (new random crop + flip each time)."""
        pick = np.random.choice(self.n_kept, size=min(n, self.n_kept), replace=False)
        images = torch.stack([data.train_set[i][0] for i in self.indices[pick]])
        return images, torch.from_numpy(self.labels[pick])

    def per_task(self):
        """How many buffer images belong to each task."""
        return np.bincount(self.labels[: self.n_kept] // 10, minlength=N_TASKS)

buffer = ReservoirBuffer(SETTINGS["buffer_size"])

## 4. One training step with replay

The training loop needs each image's dataset index (to put it in the buffer), so we use `data.get_task_loader_with_index(task)`: the same loader as before, but each batch is (images, labels, indices). It lives in `common/data.py` because DataLoader worker processes can't see classes defined inside a notebook. Then each step:

1. Take the new batch (32 images of the current task).
2. Take a buffer batch (32 random old images), if the buffer isn't empty.
3. **Join them into one batch of 64** and run one forward pass. The loss is cross-entropy over all 64.
4. Backward and update, as usual.
5. Offer the new batch's images to the buffer.

**Why this fixes output bias.** In program 03, every image in the loss belonged to the current task, so cross-entropy pushed all old classes' logits down at every step. Now half the batch is old images with old labels, so the loss also pushes old classes' logits **up** whenever an old image is shown. The output layer can no longer learn "the answer is always the newest task".

**Why one joined batch.** BatchNorm computes its statistics over the batch. With old and new images in the same batch, its running mean and variance describe all tasks, not just the newest one. (EWC couldn't protect these statistics; replay does it for free.)

In [4]:
def train_one_epoch_replay(model, loader, optimizer):
    model.train()
    total_loss, correct, seen = 0.0, 0, 0
    for images, labels, idx in loader:
        x, y = images, labels
        if len(buffer) > 0:                                  # mix in old images
            buf_x, buf_y = buffer.sample(SETTINGS["replay_batch"])
            x, y = torch.cat([images, buf_x]), torch.cat([labels, buf_y])
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(x)
            loss = F.cross_entropy(logits, y)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        buffer.add(idx, labels)                              # offer the new images to the buffer

        # Training accuracy on the new images only (the first len(labels) of the batch).
        n = labels.size(0)
        total_loss += loss.item() * n
        correct += (logits[:n].argmax(1) == y[:n]).sum().item()
        seen += n
    return total_loss / seen, correct / seen

## 5. Train task by task with replay

Same loop as programs 03 and 04: fresh SGD per task, 5 epochs, then test on every task seen so far. After each task we also print how many buffer images belong to each task. About 1.5 minutes per task, since every step now processes twice as many images.

In [ ]:
acc_cil, acc_til = new_acc_matrix(), new_acc_matrix()
buffer_history = []        # buffer images per task, after each task
start = time.time()
for task in range(N_TASKS):
    loader = data.get_task_loader_with_index(task)    # batches of (images, labels, dataset indices)
    optimizer = torch.optim.SGD(model.parameters(), lr=SETTINGS["lr"], momentum=0.9)
    for epoch in range(SETTINGS["epochs"]):
        loss, train_acc = train_one_epoch_replay(model, loader, optimizer)

    cil, til = evaluate_tasks(model, data, task, device)
    acc_cil[task, : task + 1] = cil
    acc_til[task, : task + 1] = til
    buffer_history.append(buffer.per_task())
    print(f"task {task}: train acc {train_acc*100:5.1f}%  loss {loss:.3f}  ({time.time()-start:.0f}s)"
          "\n   class-IL: " + " ".join(f"{a*100:5.1f}" for a in cil) +
          "\n   task-IL:  " + " ".join(f"{a*100:5.1f}" for a in til) +
          "\n   buffer per task: " + " ".join(str(n) for n in buffer.per_task()[: task + 1]))

## 6. Read the accuracy matrices

In [ ]:
def show(acc, title):
    print(f"{title} (%)")
    print("      " + "".join(f"{t:>6}" for t in range(N_TASKS)) + "   avg")
    for i in range(N_TASKS):
        cells = "".join("     ." if np.isnan(v) else f"{v*100:6.1f}" for v in acc[i])
        print(f"after {i}{cells}  {average_accuracy(acc, i)*100:5.1f}")
    print()

show(acc_cil, "Replay, class-IL")
show(acc_til, "Replay, task-IL")

## 7. Compare with programs 03 and 04

Loads the matrices saved by programs 03 and 04. **Done when:** replay's average accuracy clearly beats both, in class-IL.

In [ ]:
runs = {"naive (03)": np.load(RESULTS_DIR / "03_acc_matrix.npz"),
        "EWC (04)": np.load(RESULTS_DIR / "04_acc_matrix.npz"),
        "replay (05)": {"class_il": acc_cil, "task_il": acc_til}}
last = N_TASKS - 1

print(f"{'':22}" + "".join(f"{name:>14}" for name in runs))
for label, fn, key in [("final avg, class-IL", lambda a: average_accuracy(a, last), "class_il"),
                       ("final avg, task-IL",  lambda a: average_accuracy(a, last), "task_il"),
                       ("forgetting, class-IL", forgetting, "class_il"),
                       ("forgetting, task-IL",  forgetting, "task_il")]:
    print(f"{label:22}" + "".join(f"{fn(r[key])*100:13.1f}%" for r in runs.values()))

x = np.arange(N_TASKS)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, key, title in [(axes[0], "class_il", "class-IL"), (axes[1], "task_il", "task-IL")]:
    for name, r in runs.items():
        ax.plot(x, [average_accuracy(r[key], i) * 100 for i in x], marker="o", label=name)
    ax.set(title=f"Average accuracy over seen tasks, {title}", xlabel="after training task",
           ylabel="accuracy (%)", ylim=(0, 100), xticks=x)
    ax.legend()

im = axes[2].imshow(acc_cil * 100, vmin=0, vmax=100, cmap="magma")
for i in range(N_TASKS):
    for j in range(i + 1):
        v = acc_cil[i, j] * 100
        axes[2].text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7, color="black" if v > 50 else "white")
axes[2].set(title="Replay: acc[after_task][on_task], class-IL (%)", xlabel="tested on task",
            ylabel="after training task", xticks=x, yticks=x)
fig.colorbar(im, ax=axes[2], fraction=0.046)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "05_replay_vs_03_04.png", dpi=120)
plt.show()

## 8. Is the output bias gone? And what is left in the buffer?

**Output bias.** Same check as programs 03 and 04: what does the model predict for task 0's test images, and how big is each task's output layer? With replay, predictions should spread over the right task much more, and the weight norms should be much more even.

**Buffer contents.** The left plot shows how the buffer's share of each task changes over time. The last print counts how many images of **class 0** (the first class of task 0) are still in the buffer at the end. Every one of them gets replayed again and again. If we were asked to **unlearn class 0**, those images would keep teaching it back to the model. That's the link to relapse in Phase D.

In [ ]:
_, test0 = data.get_task_loaders(0)
model.eval()
with torch.no_grad():
    preds = torch.cat([model(x.to(device)).argmax(1).cpu() for x, _ in test0])
pred_task = (preds // 10).bincount(minlength=N_TASKS)
print("task 0 test images, predicted as a class of task:")
for t in range(N_TASKS):
    print(f"  task {t}: {pred_task[t].item()/len(preds)*100:5.1f}%")

w = model.fc.weight.detach().norm(dim=1).cpu().view(N_TASKS, 10).mean(1)
print("\naverage output-weight norm per task:", " ".join(f"{v:.2f}" for v in w))

hist = np.array(buffer_history)                      # rows: after task t, cols: task
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bottom = np.zeros(N_TASKS)
colors = plt.cm.viridis(np.linspace(0, 0.9, N_TASKS))
for t in range(N_TASKS):
    axes[0].bar(np.arange(N_TASKS), hist[:, t], bottom=bottom, color=colors[t], label=f"task {t}")
    bottom += hist[:, t]
axes[0].set(title="Buffer contents after each task", xlabel="after training task", ylabel="images in buffer",
            xticks=np.arange(N_TASKS))
axes[0].legend(fontsize=7, ncol=2)

axes[1].bar(np.arange(N_TASKS), w.numpy())
axes[1].set(title="Average output-weight norm per task", xlabel="task", ylabel="norm", xticks=np.arange(N_TASKS))
fig.tight_layout()
plt.show()

n_class0 = int((buffer.labels[: len(buffer)] == 0).sum())
print(f"images of class 0 ({data.class_names[0]}) still in the buffer at the end: {n_class0}")

## 9. Optional: change the buffer size

Bigger buffer, better accuracy: more of the old data is kept. Try `buffer_size` 500 and 5,000 in section 1 and run again from the top (about 15 minutes each). Every run is logged, so you can compare them later.

## 10. Log the run

In [ ]:
to_json = lambda a: [[None if np.isnan(v) else round(float(v), 4) for v in r] for r in a]
log_run("05_replay", SETTINGS,
        {"acc_class_il": to_json(acc_cil), "acc_task_il": to_json(acc_til),
         "final_avg_class_il": average_accuracy(acc_cil, N_TASKS - 1),
         "final_avg_task_il": average_accuracy(acc_til, N_TASKS - 1),
         "forgetting_class_il": forgetting(acc_cil), "forgetting_task_il": forgetting(acc_til),
         "buffer_per_task_end": buffer.per_task().tolist(),
         "minutes": round((time.time() - start) / 60, 1)})
np.savez(RESULTS_DIR / "05_acc_matrix.npz", class_il=acc_cil, task_il=acc_til)
print("logged to", RESULTS_DIR / "runs.jsonl")

## Lab notebook

*Write one paragraph here: what you saw, and what surprised you.*

Questions to answer in it:
1. How much did replay improve class-IL compared with programs 03 and 04? Why can replay fix the output bias when EWC couldn't?
2. Look at the output-weight norms. Are they more even than in programs 03 and 04? Why?
3. Each old task ends with only about 200 images in the buffer (out of 5,000). Why is that still enough to help so much?
4. Is the newest task learned as well as in program 03 (compare the diagonal)? Replay vs EWC: which costs more on new tasks?
5. Class 0 still has images in the buffer at the end. If you had to unlearn class 0 after task 5, what would happen to those images during tasks 6 to 9? (This is the relapse problem.)